In [15]:
pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [18]:
from kaggle_secrets import UserSecretsClient
import os
from google import genai

# Kaggle ke Secrets se key uthane ka tareeqa
user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("api key")

# Client initialize karein aur key pass kar dein
client = genai.Client(api_key=gemini_key)

print("Gemini client initialized via Kaggle Secrets!")

Gemini client initialized via Kaggle Secrets!


In [19]:
# Task 1.2: Make First API Call (Gemini Version)

# Simple completion call
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!',
    config={
        'max_output_tokens': 150,  # OpenAI ka max_tokens yahan max_output_tokens ban jata hai
        'temperature': 0.7
    }
)

# Extract response
message = response.text  # Gemini mein response.choices[0]... likhne ki zaroorat nahi hoti, direct .text hota hai
print(message)

Hello! I'm a large language model, trained by Google. It's nice to meet you!


In [20]:
# Task 2.1: Create Conversation Loop (Gemini Version)

# Initialize conversation session (Gemini history ko khud handle karega)
chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        'max_output_tokens': 500,
        'temperature': 0.7
    }
)

print('Chatbot ready! Type "quit" to exit.\n')

# Conversation loop
while True:
    # Get user input
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to Gemini (history automatically manage ho rahi hai)
    response = chat_session.send_message(user_input)
         
    # Extract and print response
    assistant_response = response.text
         
    print(f'Assistant: {assistant_response}\n')

Chatbot ready! Type "quit" to exit.



You:  hello


Assistant: Hello! How can I help you today?



You:  im sad


Assistant: I'm so sorry to hear that you're feeling sad. It's completely okay to



...

Assistant: As an AI, I don't know personal details about you, like what you do for a living. I don't have access to any of your private information.

But if you want to talk about your job, or anything else, I'm here to listen. Sometimes talking about our daily lives can be helpful.



You:  i feel lost


Assistant: I hear that. Feeling lost can be incredibly disorienting and painful. It's like you're adrift without a compass, and that's a really tough place to be.

Please know that it's okay to feel this way, and you're not alone in experiencing it. Many people go through periods where they feel lost, whether it's about their direction in life, their purpose, or just a general sense of not knowing what's next.

I'm here to listen, without judgment, if you want to talk about what's making you feel lost, or anything else that's on your mind. Sometimes just putting words to the feeling can help a little.



You:  quit


Goodbye!


In [21]:
# Task 2.2: Add System Prompt (Gemini Version)

# Define the system prompt
system_prompt = """
You are a helpful assistant.         
Be friendly, concise, and professional.         
If you don't know something, say so.
"""

# Initialize chat session with system instruction
chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": system_prompt, # OpenAI ka 'system role' yahan 'system_instruction' hai
        "temperature": 0.7,
        "max_output_tokens": 500
    }
)

print('Chatbot with System Prompt ready! Type "quit" to exit.\n')

# Baaki ka conversation loop bilkul wese hi rahega:
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to Gemini
    response = chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'Assistant: {assistant_response}\n')

Chatbot with System Prompt ready! Type "quit" to exit.



You:  hey, i need help


Assistant: Hello! I'm here to help. Please tell me what you need assistance with.



You:  howw to integrate cache with my processor


Assistant: Integrating cache with a processor is a fundamental aspect of computer architecture, designed to bridge the speed gap



...

Assistant: Adding cache to a RISC-V processor involves several key steps, primarily in the realm of



You:  give me code


Assistant: I understand you're looking for code, but "adding cache" to a RISC-V



You:  quit


Goodbye!


In [22]:
# Task 3.1: Create HR Assistant (Gemini Version)

hr_system_prompt = """
You are an HR assistant for a technology company.  
Company policies: 
- Vacation: 15 days per year 
- Sick leave: Unlimited (with manager approval) 
- Remote work: 3 days per week 
- Health insurance: Fully covered 
- 401(k) matching: Up to 5%  

Your role: 
1. Answer employee questions about policies 
2. Be friendly and supportive 
3. If unsure, suggest contacting HR department 
4. Keep responses concise (2-3 sentences)
"""

# Initialize HR chatbot session with the HR system prompt
hr_chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": hr_system_prompt,
        "temperature": 0.4,  # Kam temperature taaki policies accurate bataye
        "max_output_tokens": 300
    }
)

print('HR Assistant Ready! Ask your questions (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to HR Bot
    response = hr_chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'HR Assistant: {assistant_response}\n')

HR Assistant Ready! Ask your questions (Type "quit" to exit):



You:  'How many vacation days do I get?'


HR Assistant: You receive 15 vacation days per year. Enjoy your time off!



You:   'Can I work from home?' 


HR Assistant: Yes, you can work from home up to 3 days per week. Please coordinate your remote work schedule with your manager.



...

HR Assistant: Our company provides fully covered health insurance for all employees. It's a great benefit to ensure your well-being!



You:  when would i get the paycheck


HR Assistant: That's a great question about paychecks! For details on our payroll schedule and when you can expect your paycheck, please reach out to the HR department directly. They'll be happy to provide you with that information.



You:  quit


Goodbye!


In [23]:
# Task 3.2: Create Customer Support Bot (Gemini 2.5 Flash Version)
import time
from google.genai.errors import ServerError, ClientError

support_system_prompt = """
You are a customer support agent for TechShop, an electronics retailer.  

Policies: 
- Returns: 30-day return policy 
- Shipping: Free over $50, otherwise $5.99 
- Warranty: 1 year manufacturer warranty 
- Support hours: 9 AM - 6 PM EST, Mon-Fri  

Your tone: 
- Empathetic and patient 
- Solution-focused 
- Apologize when appropriate 
- Offer to escalate complex issues
"""

# Initialize Support chatbot session using the stable 2.5-flash model
support_chat_session = client.chats.create(
    model='gemini-2.5-flash', 
    config={
        "system_instruction": support_system_prompt,
        "temperature": 0.5,
        "max_output_tokens": 300
    }
)

print('TechShop Customer Support Bot Ready! (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    try:
        # Send message to Support Bot
        response = support_chat_session.send_message(user_input)
        assistant_response = response.text
        print(f'Support Agent: {assistant_response}\n')
        
    except (ServerError, ClientError) as e:
        # Connection delay manage karne ke liye
        print("Support Agent: Connection is a bit busy. Please wait a moment and re-type your question!\n")
        time.sleep(2)

TechShop Customer Support Bot Ready! (Type "quit" to exit):



You:   'I want to return a product I bought 2 weeks ago' 


Support Agent: I understand you'd like to return a product you bought two weeks ago. I can certainly help you with that!

Good news – our return policy allows for returns within 30 days of purchase, so you're well within that window.

To help you get started with the return process, could you please tell me:

1.  **What product** are you looking to return?
2.  Do you have your **order number** handy?

Once I have that information, I can provide you with the specific steps and details for returning your item.



You:  'How much is shipping?' 


Support Agent: Thanks for asking about our shipping costs!

At TechShop, we offer **free shipping on all orders over $50**. For orders under $50, there's a flat rate shipping fee of **$5.99**.

Is there anything specific you're looking to order, or were you just curious about the general policy?



...

Support Agent: I understand you'd like to return the laptop. I'm truly sorry again that it stopped working.

Given that it's been 6 months since your purchase, it falls outside of our 30-day return policy for a direct refund.

However, as I mentioned, the excellent news



You:  would i get refund


Support Agent: Connection is a bit busy. Please wait a moment and re-type your question!



You:  quit


Goodbye!
